# randn-like-noise-source — worked example 2: GAN generator samples a batch of latent vectors

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `randn-like-noise-source`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In a GAN generator, the input latent vector `z` is sampled from a standard normal distribution with shape `(batch_size, latent_dim)`. Using `torch.randn_like(template)` where `template` is a pre-allocated tensor of the right shape guarantees that the noise lands on the correct device (GPU or CPU) without any explicit `.to(device)` call. This is especially important when the generator is run inside a multi-GPU training loop.

## Worked solution

**Step 1 — Create a shape template.** We create `template = torch.zeros(batch_size, latent_dim, device=device)`. This serves as the shape and device reference; its values don't matter.

**Step 2 — Sample the noise.** `z = torch.randn_like(template)` generates a `(batch_size, latent_dim)` standard-normal tensor on the same device as `template`. No `.to(device)` needed.

**Step 3 — Feed to generator.** The generator model receives `z` and produces synthetic images. Because `z` is already on the right device, the forward pass runs without device mismatch errors.

**Step 4 — Verify properties.** We check shape, dtype, and (for CPU testing) that the values are approximately standard normal by checking mean and std.

In [ ]:
import torch as t

def sample_gan_latent(batch_size: int, latent_dim: int, device=None) -> t.Tensor:
    """Sample a batch of GAN latent vectors using randn_like for device safety."""
    if device is None:
        device = t.device('cpu')
    template = t.zeros(batch_size, latent_dim, device=device)
    return t.randn_like(template)

# Exercise: sample a batch of 16 latent vectors of dim 100
t.manual_seed(99)
batch_size, latent_dim = 16, 100
z = sample_gan_latent(batch_size, latent_dim)

print(f'Shape: {z.shape}')          # (16, 100)
print(f'Dtype: {z.dtype}')          # torch.float32
print(f'Mean:  {z.mean():.3f}')     # ~0.0
print(f'Std:   {z.std():.3f}')      # ~1.0
print(f'Device: {z.device}')        # cpu